# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Eman123-123/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

I use pre-decision page-level signals such as impressions, average position, CTR, content age, and update age. Numeric missing values are handled with median filling. Categorical values are handled with one-hot encoding when present. The target label is not included in the feature vector.

In [24]:
# Build the feature vector

import pandas as pd
import numpy as np

target_col = "trend_direction"

candidate_features = [
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "engaged_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

# Keep only features that actually exist
features = [c for c in candidate_features if c in df.columns]

X = df[features].copy()

# Fill missing numeric values with median
for col in X.select_dtypes(include=np.number).columns:
    X[col] = X[col].fillna(X[col].median())

# One-hot encode categorical columns if any are present
X = pd.get_dummies(X, drop_first=True)

print("Selected features:", features)
print("Feature matrix shape:", X.shape)
print("Target excluded:", target_col not in X.columns)

Selected features: ['impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'engaged_sessions_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']
Feature matrix shape: (30000, 17)
Target excluded: True


## 2. Feature notes (meaning, missing, categorical, available-when?)
Impressions_90d measures search visibility over the previous 90 days. Clicks_90d measures clicks over the same period. Pageviews_90d and sessions_90d measure observed traffic activity. Engaged_sessions_90d measures engaged visits. Days_with_impressions and days_with_sessions measure how consistently the page received visibility or sessions.

Impressions_last_30d, clicks_last_30d, and sessions_last_30d describe recent activity. Content_age_days measures page age, while days_since_last_update measures freshness. CTR measures click-through rate, avg_position measures observed search position, engagement_rate measures engagement, scroll_rate measures scrolling activity, and ai_traffic_pct measures the share of traffic from AI sources.

Numeric missing values are filled with the median. No categorical features are used in this feature vector. All selected features are intended to be available before the prediction/review decision. The target trend_direction is not used as a feature.
*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [25]:
# Check feature availability and missing values

feature_notes = pd.DataFrame({
    "feature": features,
    "missing_values": [int(df[c].isna().sum()) for c in features],
    "dtype": [str(df[c].dtype) for c in features],
    "available_before_prediction": [True] * len(features)
})

print(feature_notes.to_string(index=False))

               feature  missing_values   dtype  available_before_prediction
       impressions_90d               0   int64                         True
            clicks_90d               0   int64                         True
         pageviews_90d               0   int64                         True
          sessions_90d               0   int64                         True
  engaged_sessions_90d               0   int64                         True
 days_with_impressions               0   int64                         True
    days_with_sessions               0   int64                         True
  impressions_last_30d               0   int64                         True
       clicks_last_30d               0   int64                         True
     sessions_last_30d               0   int64                         True
      content_age_days               0   int64                         True
days_since_last_update               0   int64                         True
            

## 3. The leakage hunt
I checked the feature vector for direct label leakage and future-looking fields. The target trend_direction and trend_pct are excluded because they describe the outcome being predicted. I also excluded fields that could directly encode the outcome or use information from after the prediction point. The selected features are pre-decision activity, content, freshness, and engagement signals. Missing scroll_rate values are handled with median filling.
*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [26]:
# Leakage check

target_related = [
    "trend_direction",
    "trend_pct"
]

future_or_outcome_fields = [
    c for c in df.columns
    if c in target_related
]

feature_leakage = [
    c for c in X.columns
    if c in target_related
]

print("Target-related columns in raw data:", future_or_outcome_fields)
print("Target-related columns in feature matrix:", feature_leakage)
print("Leakage check passed:", len(feature_leakage) == 0)

# Check that the target is not part of selected features
assert target_col not in features
assert "trend_pct" not in features

print("Assertions passed: target and trend_pct are excluded.")

Target-related columns in raw data: ['trend_direction', 'trend_pct']
Target-related columns in feature matrix: []
Leakage check passed: True
Assertions passed: target and trend_pct are excluded.


## 4. What I excluded and why
I excluded trend_direction because it is the target label being predicted. I excluded trend_pct because it directly describes the outcome trend and could leak target information. I excluded content_id and client_id because they are identifiers rather than meaningful pre-decision signals and may create memorization or privacy risk. I also excluded provider_used and model_used because they are system/product metadata rather than page-level signals needed for this decision.
*The list of fields you refused to use — with one line of why each.*

In [27]:
# Fields excluded from the feature vector and the reason

excluded_fields = {
    "trend_direction": "Target label; must not be used as a feature.",
    "trend_pct": "Direct outcome/trend information; potential target leakage.",
    "content_id": "Identifier; not a meaningful predictive signal and may create memorization/privacy risk.",
    "client_id": "Identifier; excluded for privacy and memorization risk.",
    "provider_used": "System/product metadata rather than a page-level pre-decision signal.",
    "model_used": "System/product metadata rather than a page-level pre-decision signal."
}

excluded_check = pd.DataFrame(
    list(excluded_fields.items()),
    columns=["excluded_field", "reason"]
)

print(excluded_check.to_string(index=False))

print("\nExcluded fields are not in selected features:")
for field in excluded_fields:
    print(field, "->", field not in features)

 excluded_field                                                                                   reason
trend_direction                                             Target label; must not be used as a feature.
      trend_pct                              Direct outcome/trend information; potential target leakage.
     content_id Identifier; not a meaningful predictive signal and may create memorization/privacy risk.
      client_id                                  Identifier; excluded for privacy and memorization risk.
  provider_used                    System/product metadata rather than a page-level pre-decision signal.
     model_used                    System/product metadata rather than a page-level pre-decision signal.

Excluded fields are not in selected features:
trend_direction -> True
trend_pct -> True
content_id -> True
client_id -> True
provider_used -> True
model_used -> True


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.